In [45]:
!pip install earthaccess asf_search nisar-pytools pystac-client planetary-computer rasterio geopandas xarray dask requests
!pip install rioxarray rasterio pandas numpy scikit-learn matplotlib

In [46]:
!rm -rf /kaggle/working/*
print("Working directory cleared!")

Working directory cleared!


In [47]:
import os
import requests
import json
import numpy as np
import h5py
import dask.array as da
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt
import earthaccess
import asf_search as asf
from sklearn.cluster import KMeans
from kaggle_secrets import UserSecretsClient

# Fetch your secrets
user_secrets = UserSecretsClient()
os.environ["EARTHDATA_USERNAME"] = user_secrets.get_secret("EARTHDATA_USERNAME")
os.environ["EARTHDATA_PASSWORD"] = user_secrets.get_secret("EARTHDATA_PASSWORD")
BHOONIDHI_USER = user_secrets.get_secret("BHOONIDHI_USER")
BHOONIDHI_PASS = user_secrets.get_secret("BHOONIDHI_PASS")

# NASA Auth
earthaccess.login(strategy="environment")

# ISRO Bhoonidhi Auth
bhoonidhi_auth_url = "https://bhoonidhi-api.nrsc.gov.in/auth/token"
auth_payload = {"userId": BHOONIDHI_USER, "password": BHOONIDHI_PASS, "grant_type": "password"}
headers = {"Content-Type": "application/json"}

response = requests.post(bhoonidhi_auth_url, json=auth_payload, headers=headers)
if response.status_code == 200:
    bhoonidhi_token = response.json().get("access_token")
    print("Bhoonidhi S-Band Auth Successful! Token generated.")
else:
    print(f"Bhoonidhi Auth Failed. Status Code: {response.status_code}")

# Define a Dual-Band Focus Area
study_bbox = (91.5, 26.0, 92.5, 26.5) # Min Lon, Min Lat, Max Lon, Max Lat
start_date = "2026-07-01T00:00:00Z"
end_date = "2026-08-30T00:00:00Z"

# Directory for Phase 2 frontend assets
export_dir = "/kaggle/working/frontend_assets"
os.makedirs(export_dir, exist_ok=True)
json_path = os.path.join(export_dir, "scientist_timeseries.json")

Bhoonidhi S-Band Auth Successful! Token generated.


In [48]:
def process_and_export_scientific_insights(ds, study_bbox, platform_name, scene_id):
    print(f"Preparing spatial metadata and reprojecting {platform_name}...")
    
    # 1. Standardize coordinate names for rioxarray
    x_name = 'x' if 'x' in ds.coords else 'xCoordinates'
    y_name = 'y' if 'y' in ds.coords else 'yCoordinates'
    ds = ds.rename({x_name: 'x', y_name: 'y'})
    ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y")
    
    # 2. Determine native projection and Reproject to Lat/Lon
    native_epsg = 32646 if ds.x.values[0] > 180 or ds.x.values[0] < -180 else 4326
    ds = ds.rio.write_crs(f"EPSG:{native_epsg}")
    ds_small = ds.isel(x=slice(None, None, 4), y=slice(None, None, 4)) # Downsample for RAM safety
    ds_latlon = ds_small.rio.reproject("EPSG:4326")
    
    # 3. Crop safely using Lat/Lon bounding box
    min_lon, min_lat, max_lon, max_lat = study_bbox
    cropped_ds = ds_latlon.sel(x=slice(min_lon, max_lon), y=slice(max_lat, min_lat)).compute()
    if cropped_ds.x.size == 0 or cropped_ds.y.size == 0:
        cropped_ds = ds_latlon.compute()
    
    if 'HHHH' in cropped_ds and 'HVHV' in cropped_ds:
        print("Calculating Scientific Metrics (dB and RVI)...")
        # Convert to Decibels
        cropped_ds['HHHH_dB'] = 10 * np.log10(cropped_ds['HHHH'].where(cropped_ds['HHHH'] > 0) + 1e-10)
        cropped_ds['HVHV_dB'] = 10 * np.log10(cropped_ds['HVHV'].where(cropped_ds['HVHV'] > 0) + 1e-10)
        
        # Calculate Radar Vegetation Index (RVI)
        cropped_ds['RVI'] = 4 * cropped_ds['HVHV'] / (cropped_ds['HHHH'] + cropped_ds['HVHV'] + 1e-10)
        
        # --- STATE-OF-THE-ART ML CLUSTERING ---
        print("Running ML Unsupervised Segmentation...")
        hh_flat = cropped_ds['HHHH_dB'].values.flatten()
        rvi_flat = cropped_ds['RVI'].values.flatten()
        
        # Clean data for ML
        valid_mask = ~np.isnan(hh_flat) & ~np.isnan(rvi_flat) & np.isfinite(hh_flat) & np.isfinite(rvi_flat)
        features = np.column_stack((hh_flat[valid_mask], rvi_flat[valid_mask]))
        
        if len(features) > 0:
            kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
            predictions = kmeans.fit_predict(features)
            
            ml_map = np.full(hh_flat.shape, np.nan)
            ml_map[valid_mask] = predictions
            cropped_ds['ML_Class'] = (("y", "x"), ml_map.reshape(cropped_ds['HHHH_dB'].shape))
            
            # --- DATA VISUALIZATION ---
            plt.figure(figsize=(14, 6))
            plt.subplot(1, 2, 1)
            cropped_ds['RVI'].plot(cmap='viridis', vmin=0, vmax=1.2)
            plt.title(f'{platform_name} - Radar Vegetation Index')
            plt.axis('off')
            
            plt.subplot(1, 2, 2)
            cropped_ds['ML_Class'].plot(cmap='Set1')
            plt.title('ML Surface Segmentation (3 Classes)')
            plt.axis('off')
            
            vis_path = os.path.join(export_dir, f"Vis_{platform_name}_{scene_id}.png")
            plt.savefig(vis_path)
            plt.close()
            print(f"Saved Visualization to {vis_path}")
            
            # --- EXPORT TO COG FOR 3D GLOBE ---
            cropped_ds = cropped_ds.rio.write_crs("EPSG:4326")
            rvi_tiff = os.path.join(export_dir, f"{platform_name}_RVI_{scene_id}.tif")
            ml_tiff = os.path.join(export_dir, f"{platform_name}_ML_{scene_id}.tif")
            cropped_ds['RVI'].rio.to_raster(rvi_tiff, driver="COG", compress="DEFLATE")
            cropped_ds['ML_Class'].rio.to_raster(ml_tiff, driver="COG", compress="DEFLATE")
            
            # --- SCIENTIFIC DASHBOARD HYPOTHESIS ENGINE ---
            mean_rvi = float(cropped_ds['RVI'].mean().values)
            mean_hh = float(cropped_ds['HHHH_dB'].mean().values)
            
            # Hypothesize drivers based on dielectric constant and roughness physics
            if mean_rvi < 0.3 and mean_hh < -15.0:
                hypothesis = "High likelihood of flood inundation or surface water accumulation. Low roughness and high dielectric constant observed."
            elif mean_rvi < 0.4 and mean_hh > -10.0:
                hypothesis = "Potential deforestation or severe selective logging. Vegetation structure loss detected alongside high double-bounce ground scattering."
            else:
                hypothesis = "Stable natural processes. Normal vegetative volume scattering observed."
                
            dashboard_data = {
                "platform": platform_name,
                "scene_id": str(scene_id),
                "mean_rvi": round(mean_rvi, 3),
                "mean_roughness_db": round(mean_hh, 2),
                "scientific_hypothesis": hypothesis,
                "ml_clusters_detected": 3
            }
            
            if os.path.exists(json_path):
                with open(json_path, 'r') as f: timeseries = json.load(f)
            else: timeseries = []
            
            timeseries.append(dashboard_data)
            with open(json_path, 'w') as f: json.dump(timeseries, f, indent=4)
            print("Appended metrics and hypothesis to Scientist Dashboard JSON.")

In [49]:
print("\nSearching for NASA NISAR L-Band GCOV products...")
l_band_results = earthaccess.search_data(
    short_name="NISAR_L2_GCOV_PROVISIONAL_V1", 
    bounding_box=study_bbox,
    temporal=(start_date, end_date),
    count=3 
)

if l_band_results:
    l_band_files = earthaccess.open(l_band_results)
    for i, file_stream in enumerate(l_band_files):
        print(f"\nProcessing L-Band Scene {i+1}...")
        try:
            ds = xr.open_dataset(file_stream, engine='h5netcdf', group='/science/LSAR/GCOV/grids/frequencyA', backend_kwargs={'phony_dims': 'sort'})
            process_and_export_scientific_insights(ds, study_bbox, "NASA_L_Band", f"scene_{i}")
            ds.close() 
        except Exception as e:
            print(f"Error reading scene {i+1}: {e}")
else:
    print("No L-Band data found.")


Searching for NASA NISAR L-Band GCOV products...


QUEUEING TASKS | :   0%|          | 0/3 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/3 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/3 [00:00<?, ?it/s]


Processing L-Band Scene 1...
Preparing spatial metadata and reprojecting NASA_L_Band...
Calculating Scientific Metrics (dB and RVI)...
Running ML Unsupervised Segmentation...
Saved Visualization to /kaggle/working/frontend_assets/Vis_NASA_L_Band_scene_0.png
Appended metrics and hypothesis to Scientist Dashboard JSON.

Processing L-Band Scene 2...
Preparing spatial metadata and reprojecting NASA_L_Band...
Calculating Scientific Metrics (dB and RVI)...
Running ML Unsupervised Segmentation...
Saved Visualization to /kaggle/working/frontend_assets/Vis_NASA_L_Band_scene_1.png
Appended metrics and hypothesis to Scientist Dashboard JSON.

Processing L-Band Scene 3...
Preparing spatial metadata and reprojecting NASA_L_Band...


/usr/local/lib/python3.13/dist-packages/rasterio/warp.py:390: ComplexWarning: Casting complex values to real discards the imaginary part
  dest = _reproject(
/usr/local/lib/python3.13/dist-packages/rasterio/warp.py:390: ComplexWarning: Casting complex values to real discards the imaginary part
  dest = _reproject(
/usr/local/lib/python3.13/dist-packages/rasterio/warp.py:390: ComplexWarning: Casting complex values to real discards the imaginary part
  dest = _reproject(
/usr/local/lib/python3.13/dist-packages/rasterio/warp.py:390: ComplexWarning: Casting complex values to real discards the imaginary part
  dest = _reproject(
/usr/local/lib/python3.13/dist-packages/rasterio/warp.py:390: ComplexWarning: Casting complex values to real discards the imaginary part
  dest = _reproject(
/usr/local/lib/python3.13/dist-packages/rasterio/warp.py:390: ComplexWarning: Casting complex values to real discards the imaginary part
  dest = _reproject(


Calculating Scientific Metrics (dB and RVI)...
Running ML Unsupervised Segmentation...
Saved Visualization to /kaggle/working/frontend_assets/Vis_NASA_L_Band_scene_2.png
Appended metrics and hypothesis to Scientist Dashboard JSON.


In [50]:
# print("\nSearching for NASA NISAR L-Band GCOV products...")

# l_band_results = earthaccess.search_data(
#     short_name="NISAR_L2_GCOV_PROVISIONAL_V1", 
#     bounding_box=study_bbox,
#     temporal=(start_date, end_date),
#     count=3 
# )

# print(f"Found {len(l_band_results)} L-Band scenes.")

# if l_band_results:
#     print("Streaming directly into memory...")
#     l_band_files = earthaccess.open(l_band_results)
    
#     for i, file_stream in enumerate(l_band_files):
#         print(f"\nProcessing L-Band Scene {i+1}...")
#         try:
#             ds = xr.open_dataset(
#                 file_stream, 
#                 engine='h5netcdf',
#                 group='/science/LSAR/GCOV/grids/frequencyA',
#                 backend_kwargs={'phony_dims': 'sort'}
#             )
            
#             # --- PHASE 2: SCIENTIFIC PROCESSING (L-BAND) ---
#             print("Preparing spatial metadata and reprojecting...")
#             x_name = 'x' if 'x' in ds.coords else 'xCoordinates'
#             y_name = 'y' if 'y' in ds.coords else 'yCoordinates'
            
#             # 1. Standardize coordinate names for rioxarray
#             ds = ds.rename({x_name: 'x', y_name: 'y'})
#             ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y")
            
#             # 2. Determine the native projection
#             # If coordinates are > 180, it is in meters (UTM). We assign Assam's UTM Zone (EPSG:32646)
#             native_epsg = 32646 if ds.x.values[0] > 180 or ds.x.values[0] < -180 else 4326
#             ds = ds.rio.write_crs(f"EPSG:{native_epsg}")
            
#             # 3. Downsample slightly to allow in-memory reprojection without crashing RAM
#             # (Takes every 4th pixel, which is perfect for web-app performance)
#             ds_small = ds.isel(x=slice(None, None, 4), y=slice(None, None, 4))
            
#             # 4. Reproject to Lat/Lon (EPSG:4326) so we can slice with degrees!
#             print("Reprojecting to Web-Ready Lat/Lon (EPSG:4326)...")
#             ds_latlon = ds_small.rio.reproject("EPSG:4326")
            
#             # 5. Crop safely using Lat/Lon bounding box
#             min_lon, min_lat, max_lon, max_lat = study_bbox
#             cropped_ds = ds_latlon.sel(x=slice(min_lon, max_lon), y=slice(max_lat, min_lat)).compute()
            
#             # SAFEGUARD: If the scene doesn't perfectly overlap the box, use the full scene
#             if cropped_ds.x.size == 0 or cropped_ds.y.size == 0:
#                 print("Warning: Study box is outside this specific scene's footprint. Using full scene.")
#                 cropped_ds = ds_latlon.compute()
            
#             if 'HHHH' in cropped_ds and 'HVHV' in cropped_ds:
#                 print("Calculating Scientific Metrics (dB and RVI)...")
#                 # Convert to dB
#                 cropped_ds['HHHH_dB'] = 10 * np.log10(cropped_ds['HHHH'].where(cropped_ds['HHHH'] > 0) + 1e-10)
#                 cropped_ds['HVHV_dB'] = 10 * np.log10(cropped_ds['HVHV'].where(cropped_ds['HVHV'] > 0) + 1e-10)
                
#                 # Calculate Radar Vegetation Index (RVI)
#                 cropped_ds['RVI'] = 4 * cropped_ds['HVHV'] / (cropped_ds['HHHH'] + cropped_ds['HVHV'] + 1e-10)
                
#                 # Export COG for 3D Earth Web App
#                 tiff_path = os.path.join(export_dir, f"L_Band_RVI_scene_{i}.tif")
#                 cropped_ds['RVI'].rio.to_raster(tiff_path, driver="COG", compress="DEFLATE")
#                 print(f"Exported Web-Ready COG to {tiff_path}")
            
#             ds.close() 
            
#         except Exception as e:
#             print(f"Error reading scene {i+1}: {e}")
            
#     print("\nNASA L-Band pipeline complete!")
# else:
#     print("No L-Band data found.")

In [51]:
print("\n--- Re-Authenticating with Bhoonidhi to ensure fresh token ---")
auth_response = requests.post(bhoonidhi_auth_url, json=auth_payload, headers={"Content-Type": "application/json"})
if auth_response.status_code == 200:
    bhoonidhi_token = auth_response.json().get("access_token")
    headers["Authorization"] = f"Bearer {bhoonidhi_token}"

bhoonidhi_search_url = "https://bhoonidhi-api.nrsc.gov.in/data/search"
search_payload = {"collections": ["NISAR_SSAR_GCOV", "NISAR_SSAR-Beta_GCOV"], "bbox": study_bbox, "datetime": f"{start_date}/{end_date}", "limit": 3}

s_band_response = requests.post(bhoonidhi_search_url, headers=headers, json=search_payload)
try: s_band_features = s_band_response.json().get("features", [])
except: s_band_features = []

if len(s_band_features) == 0:
    search_payload.pop("bbox", None)
    search_payload.pop("datetime", None)
    s_band_response = requests.post(bhoonidhi_search_url, headers=headers, json=search_payload)
    s_band_features = s_band_response.json().get("features", [])[:3]

download_path = "/kaggle/working/temp_nisar"
os.makedirs(download_path, exist_ok=True)

for i, feature in enumerate(s_band_features):
    feature_id = feature.get("id")
    collection_id = feature.get("collection")
    if not feature_id or not collection_id: continue
        
    file_name = feature_id + ".h5"
    file_path = os.path.join(download_path, file_name)
    download_url = f"https://bhoonidhi-api.nrsc.gov.in/download?id={feature_id}&collection={collection_id}"
    
    print(f"\nFetching temporary S-Band file via API: {file_name}")
    with requests.get(download_url, headers=headers, stream=True, allow_redirects=True) as r:
        if r.status_code != 200: continue
        with open(file_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
    
    if (os.path.getsize(file_path) / (1024 * 1024)) < 5.0:
        os.remove(file_path)
        continue 

    try:
        with h5py.File(file_path, 'r') as f:
            freq_a = f['/science/SSAR/GCOV/grids/frequencyA']
            y_coords, x_coords = freq_a['yCoordinates'][:], freq_a['xCoordinates'][:]
            
            bands_to_load = [b for b in ['HHHH', 'HVHV'] if b in freq_a]
            if not bands_to_load: bands_to_load = [list(freq_a.keys())[0]]
            
            data_vars = {}
            for b in bands_to_load:
                lazy_data = da.from_array(freq_a[b], chunks=(2000, 2000))
                data_vars[b] = (["y", "x"], lazy_data)
                
            ds = xr.Dataset(data_vars, coords={"y": y_coords, "x": x_coords})
            process_and_export_scientific_insights(ds, study_bbox, "ISRO_S_Band", feature_id)
            ds.close()
            
    except Exception as e:
        print(f"Error processing {file_name}: {e}")
        
    if os.path.exists(file_path):
        os.remove(file_path)


--- Re-Authenticating with Bhoonidhi to ensure fresh token ---

Fetching temporary S-Band file via API: NISAR_S2_PR_GCOV_028_141_A_015_3700_DHNA_A_20260822T231345_20260822T231422_P00500_M_F_I_001.h5
Preparing spatial metadata and reprojecting ISRO_S_Band...
Calculating Scientific Metrics (dB and RVI)...
Running ML Unsupervised Segmentation...
Saved Visualization to /kaggle/working/frontend_assets/Vis_ISRO_S_Band_NISAR_S2_PR_GCOV_028_141_A_015_3700_DHNA_A_20260822T231345_20260822T231422_P00500_M_F_I_001.png
Appended metrics and hypothesis to Scientist Dashboard JSON.

Fetching temporary S-Band file via API: NISAR_S2_PR_GCOV_028_091_D_075_3700_CRNA_A_20260819T123347_20260819T123423_P00500_M_F_I_001.h5
Preparing spatial metadata and reprojecting ISRO_S_Band...

Fetching temporary S-Band file via API: NISAR_S2_PR_GCOV_027_141_A_015_3700_DHNA_A_20260810T231345_20260810T231422_P00500_M_F_I_001.h5


In [52]:
print("\nPipeline complete! COGs, ML Visuals, and Analytics JSON exported to /kaggle/working/frontend_assets/")


Pipeline complete! COGs, ML Visuals, and Analytics JSON exported to /kaggle/working/frontend_assets/


In [53]:
import shutil
from IPython.display import FileLink

# 1. Name of the zip file (without the .zip extension here)
zip_name = 'nisar_frontend_assets'

# 2. The folder you want to zip
folder_to_zip = '/kaggle/working/frontend_assets' 

# 3. Create the zip file
shutil.make_archive(zip_name, 'zip', folder_to_zip)
print("Folder successfully zipped!")

# 4. Generate a clickable download link
FileLink(f'{zip_name}.zip')

Folder successfully zipped!


/kaggle/working/nisar_frontend_assets.zip